In [1]:
import pandas as pd
import geopandas as gpd

df = gpd.read_parquet('/kaggle/input/datasets/sciencekonstant/completewithcoord/complete_with_coords.parquet')

In [2]:
df = df.drop(columns=['primary_category'])

In [3]:
import pandas as pd
import geopandas as gpd
import numpy as np

def patch_from_local_file(df, places_path):
    print("1. Loading local Parquet file (Super Fast)...")
    # Read ONLY the necessary columns directly from your Kaggle dataset
    local_places = gpd.read_parquet(
        places_path, 
        columns=['primary_category', 'confidence', 'website', 'phone', 'geometry']
    )
    
    print("2. Extracting coordinates and preparing keys...")
    # Extract lat/long directly from the local geometries to match your dataset
    local_places['patch_lat'] = local_places.geometry.y
    local_places['patch_long'] = local_places.geometry.x
    
    # Round coordinates to 6 decimals (~11cm precision) to guarantee safe float merging
    local_places['lat_key'] = local_places['patch_lat'].round(6)
    local_places['lon_key'] = local_places['patch_long'].round(6)
    
    df['lat_key'] = df['lat'].round(6)
    df['lon_key'] = df['long'].round(6)
    
    # Clean category strings to ensure a perfect string match
    local_places['patch_category'] = local_places['primary_category'].astype(str).str.lower().str.strip()
    df['target_category'] = df['target_category'].astype(str).str.lower().str.strip()
    
    # Create the binary digital footprint flags
    local_places['has_website'] = local_places['website'].notna().astype(int)
    local_places['has_phone'] = local_places['phone'].notna().astype(int)
    
    print("3. Deduplicating local patch data to prevent Cartesian explosion...")
    local_places = local_places.drop_duplicates(subset=['lat_key', 'lon_key', 'patch_category'])
    
    print("4. Executing the Merge...")
    # Merge on our 3-part composite key
    final_df = df.merge(
        local_places[['lat_key', 'lon_key', 'patch_category', 'confidence', 'has_website', 'has_phone']],
        left_on=['lat_key', 'lon_key', 'target_category'],
        right_on=['lat_key', 'lon_key', 'patch_category'],
        how='left'
    )
    
    print("5. Cleaning up...")
    # Drop temporary join keys
    final_df = final_df.drop(columns=['lat_key', 'lon_key', 'patch_category'])
    
    # Fill NA values safely
    final_df['confidence'] = final_df['confidence'].fillna(final_df['confidence'].median())
    final_df['has_website'] = final_df['has_website'].fillna(0).astype(int)
    final_df['has_phone'] = final_df['has_phone'].fillna(0).astype(int)
    
    return final_df

# Run it using the EXACT path from your Kaggle notebook
PLACES_PATH = '/kaggle/input/datasets/sciencekonstant/overdata1/india_places.parquet'

# Assuming your current 2.5M row dataframe is named 'df'
df_extra1 = patch_from_local_file(df, PLACES_PATH)

1. Loading local Parquet file (Super Fast)...
2. Extracting coordinates and preparing keys...
3. Deduplicating local patch data to prevent Cartesian explosion...
4. Executing the Merge...
5. Cleaning up...


In [4]:
!pip install h3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 4.4 MB/s eta 0:00:00a 0:00:01


In [5]:
import pandas as pd
import geopandas as gpd
import h3
import os
import urllib.request
import gzip
import shutil
import gc

def append_kontur_population(df):
    KONTUR_URL = "https://geodata-eu-central-1-kontur-public.s3.amazonaws.com/kontur_datasets/kontur_population_IN_20231101.gpkg.gz"
    GZ_FILE = "/kaggle/working/kontur_india.gpkg.gz"
    GPKG_FILE = "/kaggle/working/kontur_india.gpkg"
    
    print("1. Downloading Kontur Population Data for India (~150MB)...")
    if not os.path.exists(GZ_FILE) and not os.path.exists(GPKG_FILE):
        urllib.request.urlretrieve(KONTUR_URL, GZ_FILE)
    print("   Download complete.")

    print("2. Decompressing the Database (Fixes the DataSourceError)...")
    if not os.path.exists(GPKG_FILE):
        with gzip.open(GZ_FILE, 'rb') as f_in:
            with open(GPKG_FILE, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        os.remove(GZ_FILE)
    print("   Decompression complete.")

    print("3. Assigning H3 Hexagons (Resolution 8) to Target Data...")
    df['h3_res8'] = [
        h3.latlng_to_cell(lat, lon, 8) 
        for lat, lon in zip(df['lat'], df['long'])
    ]

    print("4. Loading Kontur Database...")
    kontur_gdf = gpd.read_file(GPKG_FILE, ignore_geometry=True)
    kontur_gdf['h3'] = kontur_gdf['h3'].astype(str)
    df['h3_res8'] = df['h3_res8'].astype(str)

    print("5. Executing Spatial Hash Join...")
    df_enriched = df.merge(
        kontur_gdf[['h3', 'population']], 
        left_on='h3_res8', 
        right_on='h3', 
        how='left'
    )

    print("6. Cleaning Up...")
    # SURGICAL CHANGE: We removed .fillna(0) here. 
    # True matches will have their actual number (including true 0s). 
    # Unmatched commercial zones will safely remain as NaN.
    
    # Drop temporary join keys
    df_enriched = df_enriched.drop(columns=['h3_res8', 'h3'])
    
    del kontur_gdf
    gc.collect()
    
    if os.path.exists(GPKG_FILE):
        os.remove(GPKG_FILE)
        
    print(f" Success. Population data appended to {len(df_enriched)} rows.")
    return df_enriched

# Execute the pipeline
df_extra2 = append_kontur_population(df_extra1)

1. Downloading Kontur Population Data for India (~150MB)...
   Download complete.
2. Decompressing the Database (Fixes the DataSourceError)...
   Decompression complete.
3. Assigning H3 Hexagons (Resolution 8) to Target Data...
4. Loading Kontur Database...
5. Executing Spatial Hash Join...
6. Cleaning Up...
 Success. Population data appended to 2559509 rows.


In [6]:
import pandas as pd
import h3
import urllib.request
import os

def append_meta_wealth_index(df):
    print("1. Downloading Meta Relative Wealth Index (India)...")
    # Meta hosts this publicly on HDX
    RWI_URL = "https://data.humdata.org/dataset/76f2a2ea-ba50-40f5-b79c-db95d668b843/resource/977923ab-c65a-4203-b216-e4b7483d56a5/download/ind_pak_relative_wealth_index.csv"
    RWI_FILE = "/kaggle/working/india_rwi.csv"
    
    if not os.path.exists(RWI_FILE):
        urllib.request.urlretrieve(RWI_URL, RWI_FILE)
        
    print("2. Loading Wealth Data...")
    rwi_df = pd.read_csv(RWI_FILE)
    
    print("3. Converting Wealth coordinates to H3 (Resolution 7 - ~5km macro grid)...")
    # Resolution 7 is perfect for capturing neighborhood-level wealth
    rwi_df['h3_res7'] = [
        h3.latlng_to_cell(lat, lon, 7) for lat, lon in zip(rwi_df['latitude'], rwi_df['longitude'])
    ]
    
    # Average the wealth index if multiple 2.4km points fall in the same Res 7 hexagon
    rwi_agg = rwi_df.groupby('h3_res7')['rwi'].mean().reset_index()
    rwi_agg['h3_res7'] = rwi_agg['h3_res7'].astype(str)
    
    print("4. Mapping to your 2.5 Million Businesses...")
    # Map your target DataFrame to Res 7
    df['h3_res7_wealth'] = [
        h3.latlng_to_cell(lat, lon, 7) for lat, lon in zip(df['lat'], df['long'])
    ]
    df['h3_res7_wealth'] = df['h3_res7_wealth'].astype(str)
    
    # Hash Join
    df_enriched = df.merge(
        rwi_agg, 
        left_on='h3_res7_wealth', 
        right_on='h3_res7', 
        how='left'
    )
    # Run this BEFORE the .fillna() step in the merge function
    missing_rwi = df_enriched['rwi'].isna().sum()
    total_rows = len(df_enriched)
    print(f"Missing RWI data: {missing_rwi} rows ({(missing_rwi/total_rows)*100:.1f}%)")
    
    # Fill missing wealth data with the median (mostly remote areas)
    # df_enriched['rwi'] = df_enriched['rwi'].fillna(df_enriched['rwi'].median())
    
    # Clean up
    df_enriched = df_enriched.drop(columns=['h3_res7_wealth', 'h3_res7'])
    if os.path.exists(RWI_FILE):
        os.remove(RWI_FILE)
        
    print("Meta Relative Wealth Index successfully added!")
    return df_enriched

df_extra3 = append_meta_wealth_index(df_extra2)

1. Downloading Meta Relative Wealth Index (India)...
2. Loading Wealth Data...
3. Converting Wealth coordinates to H3 (Resolution 7 - ~5km macro grid)...
4. Mapping to your 2.5 Million Businesses...
Missing RWI data: 481577 rows (18.8%)
Meta Relative Wealth Index successfully added!


In [ ]:
import pandas as pd
import rasterio
import numpy as np
import urllib.request
import os

def append_zenodo_nightlights(df):
    # The exact 2021 file you found on the Open-Earth-Monitor Zenodo page
    FILE_NAME = "nightlights.average_viirs.v21_m_500m_s_20210101_20211231_go_epsg4326_v20230318.tif"
    VIIRS_URL = f"https://zenodo.org/records/7750175/files/{FILE_NAME}"
    VIIRS_TIF = f"/kaggle/working/{FILE_NAME}"
        
    print("1. Fetching NASA VIIRS 2021 Nighttime Lights from Zenodo (~59MB)...")
    if not os.path.exists(VIIRS_TIF):
        urllib.request.urlretrieve(VIIRS_URL, VIIRS_TIF)
        
    print("2. Piercing Satellite Imagery for your Businesses...")
    # rasterio needs coordinates in (longitude, latitude) order (X, Y)
    coords = [(lon, lat) for lon, lat in zip(df['long'], df['lat'])]

    try:
        with rasterio.open(VIIRS_TIF) as src:
            # We "sample" (pierce) the image at our exact coordinates
            df['nighttime_lights'] = [val[0] for val in src.sample(coords)]
            
            # The author converted values to a 0-2000 scale. 
            # We floor any negative satellite anomalies to 0.
            # df['nighttime_lights'] = np.where(df['nighttime_lights'] < 0, 0, df['nighttime_lights'])
            df['nighttime_lights'] = df['nighttime_lights'].astype(float)
            df['nighttime_lights'] = df['nighttime_lights'].fillna(0) # Coastal pixels default to 0
            df['nighttime_lights'] = np.where(df['nighttime_lights'] < 0, 0, df['nighttime_lights'])
        
        print("   Nighttime lights successfully extracted.")
    except Exception as e:
        print(f"   Rasterio Error: {e}")

    print("3. Wiping satellite file from Kaggle disk...")
    if os.path.exists(VIIRS_TIF): 
        os.remove(VIIRS_TIF)

    return df

# Execute the extraction (Assuming your dataframe is currently df_with_rwi)
df_extra4 = append_zenodo_nightlights(df_extra3)

1. Fetching NASA VIIRS 2021 Nighttime Lights from Zenodo (~59MB)...
2. Piercing Satellite Imagery for your Businesses...


In [ ]:
# df_extra4.to_parquet('/kaggle/working/finalwithoutscore.parquet', compression='zstd')

In [ ]:
import pandas as pd

print("--- DataFrame Missing Values Report ---")
# Count missing values per column
missing_counts = df_extra4.isna().sum()

# Filter to show ONLY columns that have at least 1 missing value
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)

if missing_counts.empty:
    print("Perfect Dataset! Zero NaNs found in any column.")
else:
    # Calculate the percentage for context
    total_rows = len(df_extra4)
    missing_percentages = (missing_counts / total_rows) * 100
    
    # Create a clean display dataframe
    missing_report = pd.DataFrame({
        'Missing Count': missing_counts.map('{:,}'.format),
        'Percentage (%)': missing_percentages.map('{:.2f}%'.format)
    })
    
    print(f"Total Rows: {total_rows:,}\n")
    print(missing_report)

In [10]:
import pandas as pd

def impute_environmental_features(df):
    print("--- Executing Category-Aware Imputation ---")
    df = df.copy() # Protect the parent dataframe
    
    # 1. Check starting NaNs
    missing_pop_start = df['population'].isna().sum()
    missing_rwi_start = df['rwi'].isna().sum()
    print(f"Starting NaNs -> Population: {missing_pop_start:,} | RWI: {missing_rwi_start:,}")
    
    # 2. Impute based on Business Category peers
    print("\nImputing based on target_category medians...")
    for col in ['population', 'rwi']:
        df[col] = df.groupby('target_category')[col].transform(lambda x: x.fillna(x.median()))
        
    # 3. The Catch-All (For extreme edge cases)
    # If a category (e.g. 'spaceport') only has 1 row and it's NaN, the group median is also NaN.
    # We catch these rare leftovers with the global median.
    df['population'] = df['population'].fillna(df['population'].median())
    df['rwi'] = df['rwi'].fillna(df['rwi'].median())
    
    # 4. Verify Success
    missing_pop_end = df['population'].isna().sum()
    missing_rwi_end = df['rwi'].isna().sum()
    print(f"Remaining NaNs -> Population: {missing_pop_end} | RWI: {missing_rwi_end}")
    print("Imputation Complete!")
    
    return df

# Execute the dedicated imputation step
df_extra5 = impute_environmental_features(df_extra4)

--- Executing Category-Aware Imputation ---
Starting NaNs -> Population: 439,051 | RWI: 481,577

Imputing based on target_category medians...
Remaining NaNs -> Population: 0 | RWI: 0
Imputation Complete!


In [11]:
df_extra5.to_parquet('/kaggle/working/finalwithoutscore.parquet', compression='zstd')


In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA

def generate_explainable_score(df):
    features = ['confidence', 'has_website', 'has_phone', 'population', 'rwi', 'nighttime_lights']
    X = df[features].copy()
    
    # 1. Fill NaNs with category median
    for col in ['population', 'rwi', 'nighttime_lights']:
        X[col] = X.groupby(df['target_category'])[col].transform(lambda x: x.fillna(x.median()))
        X[col] = X[col].fillna(X[col].median()) 
        
    # 2. Standardize so binary (0/1) and continuous (0-2000) are treated equally
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # 3. Apply PCA
    pca = PCA(n_components=1, random_state=42)
    raw_score = pca.fit_transform(X_scaled).flatten()
    
    # Ensure direction is positive
    if pca.components_[0][0] < 0:
        raw_score = -raw_score
        pca.components_[0] = -pca.components_[0]
        
    # 4. Scale to exactly 0.0 to 1.0
    min_max = MinMaxScaler(feature_range=(0, 1))
    df['viability_score_0_1'] = min_max.fit_transform(raw_score.reshape(-1, 1))
    
    print("\nExplained Variance:", round(pca.explained_variance_ratio_[0] * 100, 2), "%")
    print("--- The Exact Mathematical Formula ---")
    
    # Calculate relative percentage weights for the interview
    weights = np.abs(pca.components_[0])
    weight_percentages = (weights / weights.sum()) * 100
    
    for feat, weight in zip(features, weight_percentages):
        print(f"{feat:>20} : {weight:.1f}%")
        
    return df

df_final = generate_explainable_score(df_extra5)


Explained Variance: 34.34 %
--- The Exact Mathematical Formula ---
          confidence : 8.2%
         has_website : 13.3%
           has_phone : 13.7%
          population : 20.0%
                 rwi : 21.3%
    nighttime_lights : 23.4%


In [13]:
df_final = df_final.drop(columns=['confidence', 'has_website', 'has_phone', 'population', 'rwi', 'nighttime_lights'])

In [14]:
df_final.to_parquet('/kaggle/working/finalwithscore.parquet', compression='zstd')